In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────────
DRY_RUN_OVERRIDE = ''   # 'true' / 'false'; blank = COPY_DRY_RUN in the config

In [ ]:
%run ./NB_Config_PROC_B1

In [ ]:
# ── Settings from NB_Config_PROC_B1 (section 3) ──────────────────────────────────────────
WORKERS, ONLY = COPY_WORKERS, COPY_ONLY
DRY_RUN = {'true': True, 'false': False}.get(DRY_RUN_OVERRIDE.strip().lower(), COPY_DRY_RUN)

In [ ]:
# ── Copy ────────────────────────────────────────────────────────────────────────
from concurrent.futures import ThreadPoolExecutor
from delta.tables import DeltaTable
fs = notebookutils.fs

def is_delta(path):
    return fs.exists(path) and DeltaTable.isDeltaTable(spark, path)

# SAP date columns carry pre-1582 values (0000-00-00): read and write them as they are
spark.conf.set('spark.sql.parquet.datetimeRebaseModeInRead',  'CORRECTED')
spark.conf.set('spark.sql.parquet.datetimeRebaseModeInWrite', 'CORRECTED')

def address(entry, side):
    """The abfss address of 'source' or 'target': written in the entry, or held by a Variable Library variable,
    plus '<side>_folder' when the variable holds the lakehouse root (e.g. 'Tables/dbo')."""
    if entry.get(side):
        return entry[side].strip().rstrip('/')
    if entry.get(f'{side}_vl'):
        library = notebookutils.variableLibrary.getLibrary(entry[f'{side}_vl'])
        base = str(library[entry[f'{side}_key']]).strip().rstrip('/')
        folder = (entry.get(f'{side}_folder') or '').strip('/')
        return f'{base}/{folder}' if folder else base
    return ''

def copy_table(from_path, to_path):
    """Read the source Delta table and write it as a plain Delta folder (same columns and partitioning, nothing registered)."""
    parts = list(DeltaTable.forPath(spark, from_path).detail().collect()[0]['partitionColumns'] or [])
    writer = spark.read.format('delta').load(from_path).write.format('delta').mode('overwrite')
    (writer.partitionBy(*parts) if parts else writer).save(to_path)

def copy_one(src, tgt, name):
    """(result, text) of one table: 'copied', 'skipped' or 'failed'."""
    s, t, tmp = f'{src}/{name}', f'{tgt}/{name}', f'{tgt}/{name}_cpy'
    try:
        if not is_delta(s):
            return 'skipped', f'{name:<18} not found in the source, skipped'
        if DRY_RUN:
            return 'dry', f"{name:<18} would copy ({'overwrite' if fs.exists(t) else 'new'})"
        if fs.exists(tmp):
            fs.rm(tmp, True)
        copy_table(s, tmp)                                # 1 copy next to the target table
        if not is_delta(tmp):
            raise RuntimeError('the copy is not a Delta folder')
        if fs.exists(t):
            fs.rm(t, True)                                # 2 overwrite: remove the old table ...
        fs.mv(tmp, t)                                     # 3 ... and rename the copy into its place
        return 'copied', f"{name:<18} copied, {spark.read.format('delta').load(t).count():,} rows"
    except Exception as e:
        try:
            if fs.exists(tmp):
                fs.rm(tmp, True)                          # never leave a half-made copy
        except Exception:
            pass                                          # the cleanup must not hide the real error
        return 'failed', f'{name:<18} FAILED: {str(e)[:150]}'

wanted = [w.strip().lower() for w in ONLY.split(',') if w.strip()]
entries = [e for e in COPIES if not wanted or any(e['name'].lower().startswith(w) for w in wanted)]
print(f"{'DRY RUN' if DRY_RUN else 'COPY'}: {len(entries)} entries, up to {WORKERS} tables at the same time\n")

totals = {'copied': 0, 'failed': 0, 'skipped': 0, 'dry': 0, 'entries_failed': 0, 'entries_skipped': 0}
for e in entries:
    print(f"── {e['name']}")
    try:
        src, tgt = address(e, 'source'), address(e, 'target')
    except Exception as ex:
        print(f'  address not found: {str(ex)[:150]}\n'); totals['entries_failed'] += 1; continue
    if not src or not tgt or '<' in src or '<' in tgt:
        print('  skipped: the source or target address is not filled in\n'); totals['entries_skipped'] += 1; continue
    try:
        for label, path in (('source', src), ('target', tgt)):               # a wrong address fails here, with the reason
            try:
                fs.ls(path)
            except Exception as ex:
                raise ValueError(f'{label} cannot be opened: {path}\nThe address must end in /Tables, or /Tables/dbo for a schema lakehouse. {str(ex)[:120]}')
    except ValueError as ex:
        print(f'  {ex}\n'); totals['entries_failed'] += 1; continue
    names = [t.strip() for t in e['tables'].replace('*', '').split(',') if t.strip()] or \
            sorted(n for n in (x.name.rstrip('/') for x in fs.ls(src)) if not n.endswith('_cpy') and is_delta(f'{src}/{n}'))
    print(f'  {src}\n  -> {tgt}\n  {len(names)} tables')
    with ThreadPoolExecutor(max_workers=max(1, WORKERS)) as pool:
        for result, text in pool.map(lambda n: copy_one(src, tgt, n), names):
            totals[result] += 1
            print('   ', text)
    print()

print(f"{totals['copied']} copied, {totals['failed']} failed, {totals['skipped']} not found in the source; "
      f"{totals['entries_skipped']} entries skipped (address not filled in), {totals['entries_failed']} entries could not be opened"
      + (f" ({totals['dry']} would be copied: dry run, nothing changed)" if DRY_RUN else ''))
if totals['failed'] or totals['entries_failed']:
    raise Exception('B1 copy: at least one table or entry failed, see above')

In [ ]:
# ── End of run: stop the session to release capacity (15 Nov 2026) ──────────────────────────────────────
# Same rules as end_session() in NB_Utils_Common, which this notebook does not load: skipped in a pipeline run
# (Fabric stops the session itself) and in a reference run; set END_SESSION = False in a cell above to keep it.
if globals().get('END_SESSION', True):
    try:
        _ctx = notebookutils.runtime.context
        if not (_ctx.get('isForPipeline') or _ctx.get('isReferenceRun')):
            print('Session stopped to release capacity (END_SESSION = False keeps it)')
            notebookutils.session.stop()
    except Exception as e:
        if 'STOP_SESSION_REQUEST' in str(e):
            raise    # Fabric stops the session through this marker exception: it must reach the runtime (18 Nov 2026)
        print(f'Session not stopped ({str(e)[:120]})')